In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from google.colab import files

In [ ]:
uploaded = files.upload()

Saving Sleep_Efficiency.csv to Sleep_Efficiency.csv


In [ ]:
df = pd.read_csv("./Sleep_Efficiency.csv")
df.head()

,ID,Age,Gender,Bedtime,Wakeup time,Sleep duration,Sleep efficiency,REM sleep percentage,Deep sleep percentage,Light sleep percentage,Awakenings,Caffeine consumption,Alcohol consumption,Smoking status,Exercise frequency
0,1,65,Female,2021-03-06 01:00:00,2021-03-06 07:00:00,6.0,0.88,18,70,12,0.0,0.0,0.0,Yes,3.0
1,2,69,Male,2021-12-05 02:00:00,2021-12-05 09:00:00,7.0,0.66,19,28,53,3.0,0.0,3.0,Yes,3.0
2,3,40,Female,2021-05-25 21:30:00,2021-05-25 05:30:00,8.0,0.89,20,70,10,1.0,0.0,0.0,No,3.0
3,4,40,Female,2021-11-03 02:30:00,2021-11-03 08:30:00,6.0,0.51,23,25,52,3.0,50.0,5.0,Yes,1.0
4,5,57,Male,2021-03-13 01:00:00,2021-03-13 09:00:00,8.0,0.76,27,55,18,3.0,0.0,3.0,No,3.0


In [ ]:
#fill na
df["Awakenings"] = df["Awakenings"].fillna(value=df["Awakenings"].mean())
df["Caffeine consumption"] = df["Caffeine consumption"].fillna(value=df["Caffeine consumption"].mean())
df["Alcohol consumption"] = df["Alcohol consumption"].fillna(value=df["Alcohol consumption"].mean())
df["Exercise frequency"] = df["Exercise frequency"].fillna(value=df["Exercise frequency"].mean())

df["Gender"] = df["Gender"].apply(lambda x: 1 if x == "Male" else 0)
df["Smoking status"] = df["Smoking status"].apply(lambda x: 1 if x == "Yes" else 0)

In [ ]:
#adding new column based on sleep duration, awakening, and sleep efficiency
def conditions(row):
  if row["Sleep duration"] in [7,8] and row["Awakenings"] <= 2 and row["Sleep efficiency"] >= 0.875:
    return 0
  else:
    return 1

# 0 = good sleep quality
# 1 = bad sleep quality

df["sleep quality"] = df.apply(conditions, axis=1)

In [ ]:
#drop columns
df = df.drop(columns=["ID", "Bedtime", "Wakeup time", "REM sleep percentage", "Deep sleep percentage", "Light sleep percentage", "Sleep duration",	"Sleep efficiency", "Awakenings"])

df = df.drop(df[df["Age"] < 18].index)

In [ ]:
xs = df.drop(columns="sleep quality")
ys = df["sleep quality"]

ys.value_counts()

,count
sleep quality,
1,354
0,89


In [ ]:
xs.describe()

,Age,Gender,Caffeine consumption,Alcohol consumption,Smoking status,Exercise frequency
count,443.000000,443.000000,443.000000,443.000000,443.000000,443.000000
mean,40.839729,0.514673,24.021072,1.197357,0.347630,1.827876
std,12.705828,0.500350,29.450887,1.603282,0.476756,1.409515
min,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,29.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,40.000000,1.000000,23.653396,0.000000,0.000000,2.000000
75%,52.000000,1.000000,50.000000,2.000000,1.000000,3.000000
max,69.000000,1.000000,200.000000,5.000000,1.000000,5.000000


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# skf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

accuracy = []
precisions = []
recalls = []
f1 = []

for train_index, test_index in skf.split(xs, ys):
    X_train, X_test = xs.iloc[train_index], xs.iloc[test_index]
    y_train, y_test = ys.iloc[train_index], ys.iloc[test_index]

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1.append(f1_score(y_test, y_pred))

print("Accuracy:", np.mean(accuracy))
print("Precision:", np.mean(precisions))
print("Recall:", np.mean(recalls))
print("F1 Score:", np.mean(f1))

Accuracy: 0.8014044943820225
Precision: 0.8122148445662456
Recall: 0.9774245472837023
F1 Score: 0.8871368085601894


In [ ]:
# Age 	Gender 	Caffeine consumption 	Alcohol consumption 	Smoking status 	Exercise frequency
model.feature_importances_

array([0.27406312, 0.09396434, 0.14659505, 0.13181966, 0.05725788,
       0.29629996])

model = RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42, class_weight='balanced')



Acuraccy: 0.6495880149812734
Precision: 0.4127619670361605
Recall: 0.5412307692307692
F1 Score: 0.4643114680029018

